## 1. Configuración de PROJ (opcional)
Ejecuta esta celda **antes** de importar geopandas/osmnx si te aparece el error `CRSError ... proj_create`. Ajusta la ruta a tu usuario.

In [1]:
import os
import pyproj

# Ajusta esta ruta a tu entorno virtual (verifica si tu usuario es "daniel" o "danie")
ruta_datos_proj = r"C:\Users\danie\Downloads\SIGRE_PROYECTO\venv\Lib\site-packages\pyproj\proj_dir\share\proj"

if os.path.isdir(ruta_datos_proj):
    os.environ["PROJ_DATA"] = ruta_datos_proj   # PROJ 9.1+
    os.environ["PROJ_LIB"] = ruta_datos_proj    # versiones anteriores
    pyproj.datadir.set_data_dir(ruta_datos_proj)
    print("Datos de PROJ configurados:", pyproj.datadir.get_data_dir())
else:
    print("Ruta de PROJ no encontrada; se usará la configuración por defecto.")

Datos de PROJ configurados: C:\Users\danie\Downloads\SIGRE_PROYECTO\venv\Lib\site-packages\pyproj\proj_dir\share\proj


## 2. Librerías

In [2]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import folium

from pathlib import Path
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import unary_union
from shapely.prepared import prep
from IPython.display import display

print("SIGRE: librerías cargadas correctamente")

SIGRE: librerías cargadas correctamente


## 3. Parámetros

In [3]:
CIUDAD = "Montería, Córdoba, Colombia"
MODO = "walk"

ORIGEN = "Universidad Cooperativa de Colombia, Montería, Córdoba, Colombia"
DESTINO = "Parque Simón Bolívar, Montería, Córdoba, Colombia"

# Factor de penalización para tramos dentro de zonas de riesgo
FACTOR_RIESGO = 100

ox.settings.use_cache = True
ox.settings.log_console = False

print("Ciudad seleccionada:", CIUDAD)

Ciudad seleccionada: Montería, Córdoba, Colombia


## 4. Red peatonal de OpenStreetMap
El grafo se mantiene **sin proyectar** (lat/lon). El atributo `length` ya viene en metros.

In [4]:
G = ox.graph_from_place(CIUDAD, network_type=MODO, simplify=True)

print("Nodos:", len(G.nodes))
print("Tramos:", len(G.edges))

Nodos: 13450
Tramos: 39086


## 5. Zonas de riesgo por inundación
Basadas en el estudio de amenaza de la Alcaldía (simuladas). Coordenadas en `(longitud, latitud)`.

In [5]:
# ZONA 1: Margen izquierda del río Sinú (El Dorado, Vallejo, El Poblado)
# Corregida: antes todas las latitudes eran 8.755 (área cero).
zona_rio_sinu = Polygon([
    (-75.895, 8.740),
    (-75.875, 8.740),
    (-75.875, 8.755),
    (-75.895, 8.755),
    (-75.895, 8.740)
])

# ZONA 2: Barrios del sur (El Tambo, El Rosario, Alboraya)
zona_sur = Polygon([
    (-75.870, 8.740),
    (-75.850, 8.740),
    (-75.850, 8.755),
    (-75.870, 8.755),
    (-75.870, 8.740)
])

# ZONA 3: Cercana a humedales (Nuevo Milenio, Playa Brígida)
zona_humedales = Polygon([
    (-75.885, 8.780),
    (-75.870, 8.780),
    (-75.870, 8.790),
    (-75.885, 8.790),
    (-75.885, 8.780)
])

# ZONA 4: Zona que cubre el punto medio de la ruta actual
# (entre la UCC y el Parque Simón Bolívar)
zona_centro = Polygon([
    (-75.876, 8.756),
    (-75.866, 8.756),
    (-75.866, 8.764),
    (-75.876, 8.764),
    (-75.876, 8.756)
])

zonas_riesgos = [zona_rio_sinu, zona_sur, zona_humedales, zona_centro]
gdf_riesgo = gpd.GeoDataFrame(geometry=zonas_riesgos, crs="EPSG:4326")

for i, z in enumerate(zonas_riesgos, start=1):
    assert z.is_valid and z.area > 0, f"La zona {i} no es un polígono válido"

print(f"Se definieron {len(zonas_riesgos)} zonas de riesgo en {CIUDAD}.")
print("Estas zonas están basadas en el estudio de amenaza de la Alcaldía.")

Se definieron 4 zonas de riesgo en Montería, Córdoba, Colombia.
Estas zonas están basadas en el estudio de amenaza de la Alcaldía.


## 6. Penalizar tramos que cruzan zonas de riesgo
Se crea `length_penalizada` en cada tramo. Los tramos que tocan una zona de riesgo cuestan `FACTOR_RIESGO` veces más, así el algoritmo intenta evitarlos.

In [6]:
zonas_prep = prep(unary_union(zonas_riesgos))   # geometría preparada = intersecciones rápidas

contador_riesgo = 0
for u, v, k, data in G.edges(keys=True, data=True):
    # Usar la geometría real del tramo si existe; si no, una línea recta entre nodos
    if "geometry" in data:
        linea = data["geometry"]
    else:
        linea = LineString([
            (G.nodes[u]["x"], G.nodes[u]["y"]),
            (G.nodes[v]["x"], G.nodes[v]["y"])
        ])

    en_riesgo = zonas_prep.intersects(linea)
    data["riesgo"] = en_riesgo
    data["length_penalizada"] = data["length"] * (FACTOR_RIESGO if en_riesgo else 1)
    contador_riesgo += en_riesgo

print(f"Se penalizaron {contador_riesgo} de {len(G.edges)} tramos que cruzan zonas de riesgo.")

Se penalizaron 6584 de 39086 tramos que cruzan zonas de riesgo.


## 7. Origen y destino

In [7]:
lat_origen, lon_origen = ox.geocode(ORIGEN)
lat_destino, lon_destino = ox.geocode(DESTINO)

nodo_origen = ox.distance.nearest_nodes(G, X=lon_origen, Y=lat_origen)
nodo_destino = ox.distance.nearest_nodes(G, X=lon_destino, Y=lat_destino)

print("Origen:", lat_origen, lon_origen)
print("Destino:", lat_destino, lon_destino)
print("Nodo de origen:", nodo_origen)
print("Nodo de destino:", nodo_destino)

Origen: 8.7664267 -75.868861
Destino: 8.7558976 -75.8869844
Nodo de origen: 9885664074
Nodo de destino: 531689755


## 8. Cálculo de rutas
Se calculan dos rutas: la más corta (ignora el riesgo) y la segura (usa `length_penalizada`). La distancia se reporta siempre en **metros reales** (`length`), sin penalización.

In [8]:
def distancia_ruta(grafo, ruta):
    """Suma de metros reales (atributo 'length') a lo largo de la ruta."""
    total = 0
    for u, v in zip(ruta[:-1], ruta[1:]):
        total += min(d["length"] for d in grafo[u][v].values())
    return total

def tramos_riesgo(grafo, ruta):
    """Número de tramos de la ruta que están en zona de riesgo."""
    n = 0
    for u, v in zip(ruta[:-1], ruta[1:]):
        # Buscar la arista con MENOR longitud PENALIZADA (la que el algoritmo prefirió)
        k = min(grafo[u][v], key=lambda kk: grafo[u][v][kk]["length_penalizada"])
        n += grafo[u][v][k]["riesgo"]
    return n

ruta_corta = nx.shortest_path(G, nodo_origen, nodo_destino, weight="length")
ruta_segura = nx.shortest_path(G, nodo_origen, nodo_destino, weight="length_penalizada")

d_corta, d_segura = distancia_ruta(G, ruta_corta), distancia_ruta(G, ruta_segura)
r_corta, r_segura = tramos_riesgo(G, ruta_corta), tramos_riesgo(G, ruta_segura)

print(f"Ruta más corta : {d_corta:,.0f} m | tramos en zona de riesgo: {r_corta}")
print(f"Ruta segura    : {d_segura:,.0f} m | tramos en zona de riesgo: {r_segura}")
print(f"Diferencia     : {d_segura - d_corta:+,.0f} m")

if ruta_corta == ruta_segura:
    print("\nAmbas rutas coinciden: el trayecto no cruza zonas de riesgo. "
          "Prueba otro origen/destino para ver el efecto de la penalización.")

Ruta más corta : 2,889 m | tramos en zona de riesgo: 8
Ruta segura    : 2,935 m | tramos en zona de riesgo: 0
Diferencia     : +46 m


## 9. Mapa
Gris = ruta más corta · Azul = ruta segura · Rojo = zonas de riesgo.

In [9]:
mapa = folium.Map(
    location=[lat_origen, lon_origen],
    zoom_start=14,
    tiles="OpenStreetMap"
)

# Zonas de riesgo
for i, zona in enumerate(zonas_riesgos, start=1):
    folium.Polygon(
        locations=[(lat, lon) for lon, lat in zona.exterior.coords],
        color="red",
        fill=True,
        fill_opacity=0.3,
        popup=f"Zona de riesgo {i}"
    ).add_to(mapa)

# Rutas
def dibujar(ruta, color, nombre, distancia, grosor):
    coords = [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in ruta]
    folium.PolyLine(
        coords, color=color, weight=grosor, opacity=0.85,
        tooltip=f"{nombre}: {distancia:,.0f} m"
    ).add_to(mapa)

dibujar(ruta_corta, "gray", "Ruta más corta", d_corta, 8)
dibujar(ruta_segura, "blue", "Ruta segura", d_segura, 4)

# Marcadores
folium.Marker([lat_origen, lon_origen], tooltip="Universidad Cooperativa de Colombia",
              icon=folium.Icon(color="blue")).add_to(mapa)
folium.Marker([lat_destino, lon_destino], tooltip='Coliseo Miguel "Happy" Lora',
              icon=folium.Icon(color="green")).add_to(mapa)

display(mapa)

## 10. Guardar el mapa

In [10]:
archivo_mapa = Path.cwd() / "mapa_rutas.html"
mapa.save(str(archivo_mapa))
print("Mapa guardado en:", archivo_mapa)

Mapa guardado en: c:\Users\danie\Downloads\SIGRE_PROYECTO\mapa_rutas.html
